# Overview

Building makemore with a simple MLP.


In [ ]:
import random

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F


# Load Data and Build Dataset


In [ ]:
with open("names.txt", "r") as file:
    names = file.read().splitlines()

print(names[:10])

In [ ]:
len(names)

In [ ]:
chars = ["."] + sorted(set("".join(names)))

s_to_i = {s: i for i, s in enumerate(chars)}
i_to_s = {i: s for i, s in enumerate(chars)}

print(chars)
print(s_to_i)
print(i_to_s)

In [ ]:
def build_dataset(names, block_size=3):
    X, y = [], []
    for name in names:
        context = [0] * block_size
        name += "."
        for ch in name:
            idx = s_to_i[ch]
            X.append(context)
            y.append(idx)
            context = context[1:] + [idx]

    return torch.tensor(X), torch.tensor(y)

In [ ]:
random.seed(42)
random.shuffle(names)

p1, p2 = int(len(names) * 0.8), int(len(names) * 0.9)

X_train, y_train = build_dataset(names[:p1])
X_val, y_val = build_dataset(names[p1:p2])
X_test, y_test = build_dataset(names[p2:])

In [ ]:
X_train.shape, X_train.dtype, y_train.shape, y_train.dtype

In [ ]:
X_train[:3]

In [ ]:
print(f"Train set size: {len(X_train)}")
print(f"Val set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

# Build MLP


In [ ]:
g = torch.Generator().manual_seed(2147483647)

# embedding matrix
C = torch.randn((27, 12), generator=g)

# first layer (hidden layer)
W1 = torch.randn(36, 200, generator=g)
b1 = torch.randn(200, generator=g)

# second layer (output layer)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)

params = [C, W1, b1, W2, b2]

for p in params:
    p.requires_grad = True

print(f"Total parameters: {sum(p.numel() for p in params)}")

# Find Good LR (LR Finder)


In [ ]:
lr_exps = torch.linspace(-3, 1, steps=1000).tolist()
losses = []

for i in range(1000):
    idxs = torch.randint(0, X_train.shape[0], (256,), generator=g)
    X_batch, y_batch = X_train[idxs], y_train[idxs]

    # forward pass
    emb = C[X_batch].reshape((len(X_batch), -1))
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y_batch)

    # backward pass
    for p in params:
        p.grad = None
    loss.backward()
    lr = 10 ** lr_exps[i]
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad

    # track stats
    losses.append(loss.item())

In [ ]:
plt.plot(lr_exps, losses)

# Train and Eval Functions


Run for fresh start after LR finder


In [ ]:
def build_model(block_size, embedding_size, hidden_size, generator):
    # embedding matrix
    C = torch.randn((27, embedding_size), generator=generator)

    # first layer (hidden layer)
    W1 = torch.randn(block_size * embedding_size, hidden_size, generator=generator)
    b1 = torch.randn(hidden_size, generator=generator)

    # second layer (output layer)
    W2 = torch.randn((hidden_size, 27), generator=generator)
    b2 = torch.randn(27, generator=generator)

    params = [C, W1, b1, W2, b2]

    for p in params:
        p.requires_grad = True

    return params

In [ ]:
def train(params, X, y, lr, generator, batch_size=256, num_steps=2000):
    C, W1, b1, W2, b2 = params[0], params[1], params[2], params[3], params[4]

    for _ in range(num_steps):
        # get mini-batch
        idxs = torch.randint(0, X.shape[0], (batch_size,), generator=generator)
        X_batch, y_batch = X[idxs], y[idxs]

        # forward pass
        emb = C[X_batch].reshape((len(X_batch), -1))
        h = torch.tanh(emb @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, y_batch)

        # backward pass
        for p in params:
            p.grad = None
        loss.backward()
        with torch.no_grad():
            for p in params:
                p -= lr * p.grad


def evaluate(params, X, y):
    C, W1, b1, W2, b2 = params[0], params[1], params[2], params[3], params[4]
    emb = C[X].reshape((len(X), -1))
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    return F.cross_entropy(logits, y).item()

Run code below with lr starting from 0.2 and reducing manually upon plateau. This will lead to dev set loss less than 2.1701, which was the goal of the exercise given in the video (makemore part 2).


In [ ]:
g = torch.Generator().manual_seed(2147483647)

X_train, y_train = build_dataset(names[:p1], block_size=4)
X_val, y_val = build_dataset(names[p1:p2], block_size=4)
params = build_model(block_size=4, embedding_size=12, hidden_size=200, generator=g)

In [ ]:
train(params, X_train, y_train, lr=0.02, generator=g, num_steps=20000)
print(f"Train loss: {evaluate(params, X_train, y_train)}")
print(f"Val loss: {evaluate(params, X_val, y_val)}")

# Inference


In [ ]:
C, W1, b1, W2, b2 = params

for _ in range(5):
    x = [0] * 4
    out = ""
    while True:
        emb = C[torch.tensor(x)].reshape(1, -1)
        h = torch.tanh(emb @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        idx = torch.multinomial(probs, num_samples=1).item()

        if idx == 0:
            break

        x = x[1:] + [idx]
        out += i_to_s[idx]

    print(out)